# NEXUS-KO · interface **energy** vs co-dependency (AlphaFold-Multimer + PRODIGY, GPU)

For each obligate protein pair: **predict the complex with AlphaFold-Multimer (ColabFold)**, compute a real interface **binding energy ΔG with PRODIGY** (not just residue count), and test whether interface *energy* predicts DepMap **co-dependency** — the question crude interface size failed (Spearman 0.19, p=0.29 on 34 fetched real structures).

**Everything autosaves to your Google Drive** at `MyDrive/nexus_ko/` — predicted structures, scores, and the results JSON — so a runtime disconnect doesn't lose work, and re-running **resumes** (already-predicted pairs are skipped).

**Setup:** `Runtime → Change runtime type → GPU`, then run cells top to bottom.

**Honest bounds:** protein-level *“which complexes break”* readout, validated against co-dependency — **not** the mRNA far field, **not** the perturbation wall. AF-Multimer ipTM and PRODIGY ΔG are estimates; trust rankings, not absolute values.

In [ ]:
!nvidia-smi -L  # confirm a GPU is attached (Runtime -> Change runtime type -> GPU)


## 1 · Mount Google Drive (autosave target)
All outputs land in `MyDrive/nexus_ko/`. Set `MOUNT_DRIVE=False` to run locally instead (results lost on disconnect).

In [ ]:
import os
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/nexus_ko'
else:
    WORK = '/content/nexus_ko'
AF_IN, AF_OUT = f'{WORK}/af_in', f'{WORK}/af_out'
RESULT_JSON = f'{WORK}/nexus_ko_afmultimer.json'
for d in (WORK, AF_IN, AF_OUT): os.makedirs(d, exist_ok=True)
print('autosaving everything to:', WORK)
print('  structures -> ', AF_OUT, '\n  results    -> ', RESULT_JSON)


## 2 · Install ColabFold (AF-Multimer) + PRODIGY

In [ ]:
import os
if not os.path.exists(f'{WORK}/.installed'):
    os.system('pip -q install "colabfold[alphafold]" 2>/dev/null')
    os.system('pip -q install prodigy-prot biopython scipy 2>/dev/null')
    open(f'{WORK}/.installed','w').close()
print('colabfold_batch:', os.popen('which colabfold_batch').read().strip() or 'NOT FOUND')


## 3 · Obligate pairs (embedded)
`pdb`/`real_iface`: the real structure + measured interface (residues) the sandbox already fetched, where one exists.

In [ ]:
PAIRS = [{"a": "TSC1", "b": "TSC2", "codep": 0.912, "pdb": "7DL2", "real_iface": 200}, {"a": "DEPDC5", "b": "NPRL2", "codep": 0.799, "pdb": "6CES", "real_iface": 77}, {"a": "SDHA", "b": "SDHB", "codep": 0.794, "pdb": "8GS8", "real_iface": 127}, {"a": "RNASEH2A", "b": "RNASEH2C", "codep": 0.789, "pdb": "3P56", "real_iface": 120}, {"a": "AP2M1", "b": "AP2S1", "codep": 0.781, "pdb": "6URI", "real_iface": 9}, {"a": "POLE3", "b": "POLE4", "codep": 0.776, "pdb": null, "real_iface": null}, {"a": "WDR26", "b": "YPEL5", "codep": 0.762, "pdb": "8QBN", "real_iface": 58}, {"a": "PDHA1", "b": "PDHB", "codep": 0.758, "pdb": "1NI4", "real_iface": 512}, {"a": "HSD17B10", "b": "PRORP", "codep": 0.757, "pdb": null, "real_iface": null}, {"a": "POLG", "b": "POLG2", "codep": 0.745, "pdb": "3IKM", "real_iface": 234}, {"a": "MAPKAP1", "b": "RICTOR", "codep": 0.744, "pdb": "5ZCS", "real_iface": 42}, {"a": "GATB", "b": "QRSL1", "codep": 0.736, "pdb": null, "real_iface": null}, {"a": "RGP1", "b": "RIC1", "codep": 0.736, "pdb": null, "real_iface": null}, {"a": "MIOS", "b": "WDR24", "codep": 0.733, "pdb": "7UHY", "real_iface": 145}, {"a": "KCMF1", "b": "UBR4", "codep": 0.733, "pdb": "9NWE", "real_iface": 202}, {"a": "NCAPG2", "b": "NCAPH2", "codep": 0.729, "pdb": "9F5W", "real_iface": 85}, {"a": "EED", "b": "EZH2", "codep": 0.728, "pdb": "5GSA", "real_iface": 144}, {"a": "BRD9", "b": "SMARCD1", "codep": 0.727, "pdb": null, "real_iface": null}, {"a": "EXT1", "b": "EXT2", "codep": 0.727, "pdb": "7SCH", "real_iface": 181}, {"a": "PARD3", "b": "PARD6B", "codep": 0.726, "pdb": null, "real_iface": null}, {"a": "GET1", "b": "GET3", "codep": 0.726, "pdb": "6SO5", "real_iface": 63}, {"a": "PSMG1", "b": "PSMG2", "codep": 0.725, "pdb": "8QYJ", "real_iface": 72}, {"a": "NCAPD3", "b": "NCAPH2", "codep": 0.723, "pdb": "9F5W", "real_iface": 119}, {"a": "DLST", "b": "OGDH", "codep": 0.706, "pdb": null, "real_iface": null}, {"a": "TGFBR1", "b": "TGFBR2", "codep": 0.704, "pdb": "2PJY", "real_iface": 25}, {"a": "GATB", "b": "GATC", "codep": 0.702, "pdb": null, "real_iface": null}, {"a": "MAU2", "b": "NIPBL", "codep": 0.701, "pdb": null, "real_iface": null}, {"a": "STN1", "b": "TEN1", "codep": 0.701, "pdb": "4JOI", "real_iface": 191}, {"a": "RAD51D", "b": "XRCC2", "codep": 0.697, "pdb": "8FAZ", "real_iface": 108}, {"a": "MIOS", "b": "WDR59", "codep": 0.696, "pdb": "7UHY", "real_iface": 130}, {"a": "NCAPD3", "b": "NCAPG2", "codep": 0.694, "pdb": "9F5W", "real_iface": 21}, {"a": "VPS18", "b": "VPS33A", "codep": 0.694, "pdb": null, "real_iface": null}, {"a": "FANCD2", "b": "FANCI", "codep": 0.694, "pdb": "6VAA", "real_iface": 111}, {"a": "METTL1", "b": "WDR4", "codep": 0.685, "pdb": "7U20", "real_iface": 49}, {"a": "EED", "b": "SUZ12", "codep": 0.685, "pdb": "4W2R", "real_iface": 62}, {"a": "PNPT1", "b": "SUPV3L1", "codep": 0.685, "pdb": null, "real_iface": null}, {"a": "HUS1", "b": "RAD9A", "codep": 0.683, "pdb": "3A1J", "real_iface": 55}, {"a": "EZH2", "b": "SUZ12", "codep": 0.683, "pdb": "5HYN", "real_iface": 551}, {"a": "ITGAV", "b": "ITGB5", "codep": 0.678, "pdb": null, "real_iface": null}, {"a": "RNASEH2A", "b": "RNASEH2B", "codep": 0.675, "pdb": "3P56", "real_iface": 77}, {"a": "MLST8", "b": "RICTOR", "codep": 0.675, "pdb": null, "real_iface": null}, {"a": "SDHB", "b": "SDHC", "codep": 0.674, "pdb": "8GS8", "real_iface": 81}, {"a": "PARD6B", "b": "PRKCI", "codep": 0.669, "pdb": null, "real_iface": null}, {"a": "MICOS10", "b": "MICOS13", "codep": 0.666, "pdb": null, "real_iface": null}, {"a": "PRORP", "b": "TRMT10C", "codep": 0.659, "pdb": "7ONU", "real_iface": 45}, {"a": "MAEA", "b": "WDR26", "codep": 0.657, "pdb": null, "real_iface": null}, {"a": "CBFB", "b": "RUNX1", "codep": 0.656, "pdb": "1E50", "real_iface": 332}, {"a": "COG5", "b": "COG7", "codep": 0.655, "pdb": null, "real_iface": null}, {"a": "ACTR2", "b": "ARPC4", "codep": 0.654, "pdb": "6UHC", "real_iface": 37}, {"a": "TUBD1", "b": "TUBE1", "codep": 0.653, "pdb": null, "real_iface": null}, {"a": "CABIN1", "b": "HIRA", "codep": 0.651, "pdb": null, "real_iface": null}, {"a": "SDHA", "b": "SDHC", "codep": 0.647, "pdb": "8GS8", "real_iface": 3}, {"a": "VPS39", "b": "VPS41", "codep": 0.647, "pdb": null, "real_iface": null}, {"a": "PEX26", "b": "PEX6", "codep": 0.645, "pdb": null, "real_iface": null}, {"a": "RNASEH2B", "b": "RNASEH2C", "codep": 0.642, "pdb": "3P56", "real_iface": 216}]
N_PAIRS = 20   # predict this many (no-structure pairs first, then with-structure for validation). Up to 55.
work = sorted(PAIRS, key=lambda p: (p['pdb'] is not None, -p['codep']))[:N_PAIRS]
print(f'targeting {len(work)} pairs; {sum(1 for p in work if not p["pdb"])} have no solved structure')


## 4 · Sequences (UniProt) → ColabFold multimer FASTAs (autosaved to Drive; skips already-predicted)

In [ ]:
import requests, glob
def gene_seq(g):
    r = requests.get('https://rest.uniprot.org/uniprotkb/search',
                     params={'query':f'gene_exact:{g} AND organism_id:9606 AND reviewed:true','fields':'sequence','format':'fasta'}, timeout=30)
    return ''.join(r.text.split('\n')[1:]).strip() if (r.status_code==200 and r.text.startswith('>')) else None
seqs = {}
for p in work:
    for g in (p['a'], p['b']):
        if g not in seqs: seqs[g] = gene_seq(g)
written = []
for p in work:
    name = f"{p['a']}__{p['b']}"
    if glob.glob(f'{AF_OUT}/{name}_*rank_001*.pdb'):  # already predicted in Drive -> resume-skip
        written.append((name, p)); continue
    sa, sb = seqs.get(p['a']), seqs.get(p['b'])
    if not sa or not sb or len(sa)+len(sb) > 1800: continue
    open(f'{AF_IN}/{name}.fasta','w').write(f'>{name}\n{sa}:{sb}\n')
    written.append((name, p))
print('ready:', len(written), 'pairs (FASTAs in Drive; already-predicted ones will be skipped)')


## 5 · Run AlphaFold-Multimer → structures autosave to Drive (resume-safe)

In [ ]:
# writes each predicted complex straight into AF_OUT on Drive; re-running skips finished pairs
os.system(f'colabfold_batch "{AF_IN}" "{AF_OUT}" --num-models 1 --num-recycle 3 --rank iptm 2>&1 | tail -5')
print('predictions saved to', AF_OUT)


## 6 · Interface **energy** (PRODIGY ΔG) + AF confidence — autosaved incrementally

In [ ]:
import json, glob, re, numpy as np
def top_pdb(name):
    h = sorted(glob.glob(f'{AF_OUT}/{name}_*rank_001*.pdb') + glob.glob(f'{AF_OUT}/{name}*rank_1*.pdb'))
    return h[0] if h else None
def sc(name):
    j = sorted(glob.glob(f'{AF_OUT}/{name}_*rank_001*.json') + glob.glob(f'{AF_OUT}/{name}*scores*rank_1*.json'))
    if not j: return {}
    d = json.load(open(j[0])); return {'iptm': d.get('iptm'), 'ptm': d.get('ptm'), 'pae': d.get('pae') or d.get('predicted_aligned_error')}
def prodigy_dg(pdb):
    o = os.popen(f'prodigy "{pdb}" --selection A B -q 2>/dev/null').read()
    m = re.search(r'(-?\d+\.\d+)', o.strip().split(chr(10))[-1]) if o.strip() else None
    return float(m.group(1)) if m else None
rows = []
for name, p in written:
    pdb = top_pdb(name)
    if not pdb: continue
    s = sc(name); dg = prodigy_dg(pdb); ipae = None
    try:
        la = len(seqs[p['a']]); pae = np.array(s['pae']); ipae = float((pae[:la, la:].mean()+pae[la:, :la].mean())/2)
    except Exception: pass
    rows.append({**p, 'iptm': s.get('iptm'), 'interface_pae': ipae, 'prodigy_dG': dg, 'pdb_file': os.path.basename(pdb)})
    json.dump(rows, open(RESULT_JSON,'w'), indent=1)   # AUTOSAVE to Drive after every pair
    print(f"{p['a']:9s}-{p['b']:9s} codep={p['codep']:.2f}  ipTM={s.get('iptm')}  ifacePAE={ipae}  dG={dg}")
print('\nsaved', len(rows), 'rows ->', RESULT_JSON)


## 7 · Does interface **energy** predict co-dependency? (the test size failed)

In [ ]:
import numpy as np
from scipy.stats import spearmanr
R = [r for r in rows if any(r.get(k) is not None for k in ('prodigy_dG','iptm','interface_pae'))]
for metric, sign in [('prodigy_dG', -1), ('iptm', 1), ('interface_pae', -1)]:
    v = [(r[metric], r['codep']) for r in R if r.get(metric) is not None]
    if len(v) >= 6:
        a, c = zip(*v); rho, p = spearmanr([sign*x for x in a], c)
        print(f'{metric:14s} vs co-dependency: rho={rho:+.3f} (p={p:.3f}, n={len(v)})  [sign {sign:+d}: higher=stronger interface]')
val = [(r['prodigy_dG'], r['real_iface']) for r in R if r.get('prodigy_dG') is not None and r.get('real_iface')]
if len(val) >= 5:
    a, b = zip(*val); print(f'\npredicted ΔG vs real fetched interface size: rho={spearmanr(a,b)[0]:+.3f} (n={len(val)})')
print('\nRead: if ΔG / ipTM beats the size result (rho~0.19, p=0.29), interface ENERGY sharpens obligate prediction where crude size did not.')


## 8 · Where your results are + honest bounds

- **In your Google Drive:** `MyDrive/nexus_ko/` — predicted structures in `af_out/`, energies/scores in `nexus_ko_afmultimer.json`. Persists across disconnects; re-running resumes.
- Hand `nexus_ko_afmultimer.json` back to the sandbox to fold into the NEXUS-KO record.
- **Bounds:** protein-level “which complexes break”, validated against co-dependency — not the mRNA far field, not the wall. ipTM and PRODIGY ΔG are estimates; use rankings. Small N and top-obligate selection bias apply — raise `N_PAIRS` and add matched non-obligate contacting pairs for a cleaner test.